# 多账户 Brinson 归因：三种决策视角

本教程用**同一份完全模拟的数据**比较三种方法：

1. **Macro → Micro**：账户配置，再用账户内 Brinson 解释账户超额。
2. **价值加权总组合**：先汇总持仓，再做一次总组合 Brinson。
3. **层级/决策归因**：基准 → 账户权重 → 行业权重 → 个股权重 → 个股收益。

金额是期初市值或基准名义金额；收益率、权重和效应均为十进制。数据中没有真实证券、交易、账户划转、现金流或费用。完整公式见 `docs/source/multi-account.md`。

In [ ]:
from pathlib import Path
import sys
import polars as pl
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "brinsons").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Launch Jupyter from the project root or tutorials/ directory")
sys.path.insert(0, str(ROOT / "src"))
from brinsons import (
    macro_micro_attribution, value_weighted_total_attribution,
    hierarchical_decision_attribution, link_macro_micro_attribution,
    link_hierarchical_decision_attribution, link_carino, link_grap,
    link_frongello, link_menchero, bhb,
)
DATA = ROOT / "data" / "multi_account" / "positions.csv"
positions = pl.read_csv(DATA)
print(f"Loaded {positions.height} synthetic observations from {DATA}")
display(positions.head(8))

## 0. 核对输入层级

每期应有 3 个账户 × 2 个行业 × 2 个模拟个股。组合市值和基准名义金额各为 1000 万，且账户内有明确的两侧条件权重。

In [ ]:
value_check = positions.group_by("period", "account").agg(
    pl.col("portfolio_value", "benchmark_value").sum()
).sort("period", "account")
display(value_check)
assert value_check.height == 9
assert positions.select(pl.struct("period", "account", "sector", "security").is_duplicated().any()).item() is False

## 1. Macro 账户配置 → Micro 账户内 Brinson

先将账户看作 Brinson 的类别。默认使用 BF 三效应：

$$A_a^{macro}=(W^P_a-W^B_a)(R^B_a-R_B),\quad
S_a^{macro}=W^B_a(R^P_a-R^B_a),\quad
I_a^{macro}=(W^P_a-W^B_a)(R^P_a-R^B_a).$$

随后在每个账户内按行业做 BF。整合时，账户层 $S_a^{macro}+I_a^{macro}=W^P_a(R^P_a-R^B_a)$ 由账户内三效应乘 $W^P_a$ **替代**，避免重复计算。

In [ ]:
macro_micro = macro_micro_attribution(positions)
print("Account-level macro effects")
display(macro_micro.macro.detail.select(
    "period", "account", "_ma_wp", "_ma_wb", "_ma_rp", "_ma_rb",
    "allocation", "selection", "interaction"
).head(9))
print("Within-account micro Brinson")
display(macro_micro.micro_summary.head(9))
print("Integrated total")
display(macro_micro.integrated_summary)

In [ ]:
account_check = macro_micro.integrated_detail.with_columns(
    (pl.col("macro_selection") + pl.col("macro_interaction")).alias("macro_active"),
    pl.sum_horizontal("weighted_micro_allocation", "weighted_micro_selection",
                      "weighted_micro_interaction").alias("micro_replacement")
)
assert max(abs(x) for x in account_check.select(
    pl.col("macro_active") - pl.col("micro_replacement")).to_series()) < 1e-12
display(account_check.select("period", "account", "account_allocation",
                             "macro_active", "micro_replacement", "total_effect").head(9))

可把 `column_micro_segment="security"` 传给 `macro_micro_attribution`，直接查看账户内个股 Brinson；也可分别传 `macro_attribution=bhb`、`micro_attribution=bhb` 或 `macro_effects=2`、`micro_effects=2`。

In [ ]:
stock_micro = macro_micro_attribution(positions, column_micro_segment="security",
                                     macro_attribution=bhb, micro_effects=2)
print("Stock-level micro, two effects; single-period views do not use linking")
display(stock_micro.micro_detail.select("period", "account", "security",
                                         "allocation", "selection", "interaction").head(8))

## 2. 价值加权合并为一个总组合

跨账户汇总同一行业的组合市值和基准名义金额，形成全组合权重及加权行业收益，然后调用已有的 `fachler`。账户配置决策不会单独显示，但组合与基准的总收益应与第一种方法完全一致。

In [ ]:
pooled = value_weighted_total_attribution(positions)
print("One pooled sector Brinson per period")
display(pooled.detail.select("period", "sector", "_ma_total_wp", "_ma_total_wb",
                             "allocation", "selection", "interaction").head(8))
display(pooled.summary)

## 3. 层级 / 决策归因

固定决策顺序，并逐步改变 notional portfolio：

$$N_0=R_B,\quad N_1=\text{账户权重改为组合},\quad
N_2=\text{行业条件权重改为组合},\quad
N_3=\text{个股条件权重改为组合},\quad N_4=R_P.$$

四个阶段的效应分别为 $N_1-N_0$、$N_2-N_1$、$N_3-N_2$ 和 $N_4-N_3$。每层的行效应以该层父节点基准收益为中心（BF 风格）。

In [ ]:
decision = hierarchical_decision_attribution(positions)
print("Successive notional portfolio returns")
display(decision.notionals)
print("Four decision effects")
display(decision.summary)
print("Decision-tree rows")
display(decision.detail.head(12))

### 单期决策瀑布图（明确标记：Linking: none）

In [ ]:
chosen = "2026-01"
row = decision.summary.filter(pl.col("period") == chosen).row(0, named=True)
stages = ["account_allocation", "sector_allocation", "security_allocation", "security_selection"]
labels = ["Account", "Sector", "Stock weight", "Stock selection"]
values = [row[name] * 10_000 for name in stages]
starts = [0.0]
for value in values[:-1]:
    starts.append(starts[-1] + value)
fig, ax = plt.subplots(figsize=(9, 4), layout="constrained")
for i, (start, value) in enumerate(zip(starts, values)):
    ax.bar(i, value, bottom=start, color="#2874a6" if value >= 0 else "#c0392b")
    ax.text(i, start + value, f"{value:+.2f}", ha="center", va="bottom" if value >= 0 else "top")
ax.bar(len(values), row["excess_return"] * 10_000, color="#444", label="Total excess")
ax.set_xticks(range(len(values) + 1), labels + ["Total"])
ax.axhline(0, color="black", lw=.7)
ax.set_ylabel("Basis points")
ax.set_title(f"Decision attribution waterfall — {chosen} — Linking: none")
plt.show()

## 三种方法：同一总收益、不同决策解释

In [ ]:
comparison = macro_micro.integrated_summary.select(
    "period", pl.col("account_allocation").alias("macro_account_allocation"),
    pl.col("excess_return").alias("macro_excess")
).join(pooled.summary.select(
    "period", pl.col("allocation").alias("pooled_sector_allocation"),
    pl.col("excess_return").alias("pooled_excess")
), on="period").join(decision.summary.select(
    "period", "account_allocation", "sector_allocation", "security_allocation",
    "security_selection", pl.col("excess_return").alias("decision_excess")
), on="period").sort("period")
display(comparison)
assert comparison.select(
    ((pl.col("macro_excess")-pl.col("pooled_excess")).abs() < 1e-12).all().alias("pooled_ok"),
    ((pl.col("macro_excess")-pl.col("decision_excess")).abs() < 1e-12).all().alias("decision_ok"),
).row(0) == (True, True)

## 多期链接：每次显式选择并标明方法

下表在**同一份三期数据**上比较四种算术链接。三个视角对复合后的总超额收益一致，但效应分配不同。`link_macro_micro_attribution` 和 `link_hierarchical_decision_attribution` 对各期所有可加效应采用同一个链接系数。几何链接使用不同的乘法目标，不适用于这两张可加效应表。

In [ ]:
linkers = [link_carino, link_grap, link_frongello, link_menchero]
linked_runs = {}
rows = []
for linker in linkers:
    method = linker.__name__.removeprefix("link_").title()
    macro_linked = link_macro_micro_attribution(macro_micro, linking=linker)
    decision_linked = link_hierarchical_decision_attribution(decision, linking=linker)
    pooled_linked = linker(pooled)
    linked_runs[method] = (macro_linked, decision_linked, pooled_linked)
    rows.append({
        "Linking method": method,
        "Macro→Micro excess (bp)": 10_000 * macro_linked.summary["excess_return"][0],
        "Pooled excess (bp)": 10_000 * pooled_linked.summary["excess_return"][0],
        "Decision excess (bp)": 10_000 * decision_linked.summary["excess_return"][0],
        "Macro account allocation (bp)": 10_000 * macro_linked.summary["account_allocation"][0],
        "Decision account allocation (bp)": 10_000 * decision_linked.summary["account_allocation"][0],
    })
linked_comparison = pl.DataFrame(rows)
display(linked_comparison)
assert linked_comparison.select((pl.col("Macro→Micro excess (bp)")-
                                 pl.col("Decision excess (bp)")).abs().max()).item() < 1e-8

### 多期效应对比图（每个图例均标记 linking 方法）

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), layout="constrained")
for method, (macro_linked, decision_linked, _) in linked_runs.items():
    ms = macro_linked.summary.row(0, named=True)
    ds = decision_linked.summary.row(0, named=True)
    axes[0].scatter(10_000 * ms["account_allocation"],
                    10_000 * sum(ms[x] for x in ["weighted_micro_allocation",
                                                "weighted_micro_selection", "weighted_micro_interaction"]),
                    s=75, label=f"Linking: {method}")
    axes[1].scatter(10_000 * ds["account_allocation"],
                    10_000 * (ds["sector_allocation"] + ds["security_allocation"]),
                    s=75, label=f"Linking: {method}")
axes[0].set(title="Macro→Micro: account vs within-account", xlabel="Account allocation (bp)",
            ylabel="Weighted micro effects (bp)")
axes[1].set(title="Decision tree: account vs lower-level allocation",
            xlabel="Account allocation (bp)", ylabel="Sector + stock allocation (bp)")
for ax in axes:
    ax.axhline(0, color="gray", lw=.7)
    ax.axvline(0, color="gray", lw=.7)
    ax.legend(fontsize=8)
    ax.grid(alpha=.2)
plt.show()

## 阅读结果时的边界

- 账户和行业条件基准必须存在；零基准权重的整个账户或行业没有可定义的条件收益，会明确报错。
- 账户间划转、费用、现金流、期间内变动权重需要上游计量与有效日期数据；本例使用期初名义金额，不从期末市值反推权重。
- 三种方法可有相同总收益但不同 Allocation/Selection 分配。报告须注明决策顺序和多期 linking 方法。